In [2]:

import inspect
import time
import traceback
from pathlib import Path

from pinecone import Pinecone, ServerlessSpec

from app.config import settings
from app.ingestion.loading_data import (
    PineConeIntegration,
    get_data_from_pinecone,
)


INDEX_NAME = "knowledge-base-customer-assistant"
NAMESPACE = "support-docs"
DATA_DIR = Path("data")


def create_index_if_missing(pc: Pinecone):
    """Create the index if absent, then wait until it is ready."""

    indexes = pc.list_indexes().names()

    print("Available indexes:", indexes)
    print("Requested index:", repr(INDEX_NAME))

    if INDEX_NAME not in indexes:
        print(f"\nIndex '{INDEX_NAME}' does not exist.")
        print("Creating Pinecone index...")

        pc.create_index(
            name=INDEX_NAME,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1",
            ),
        )

    print("\nWaiting for index readiness...")

    for attempt in range(90):
        description = pc.describe_index(name=INDEX_NAME)
        status = description.status

        if isinstance(status, dict):
            ready = status.get("ready", False)
        else:
            ready = getattr(status, "ready", False)

        if ready:
            print(f"Index '{INDEX_NAME}' is ready.")
            return

        time.sleep(1)

    raise TimeoutError(
        f"Index '{INDEX_NAME}' did not become ready within 90 seconds."
    )


def main():
    try:
        print("1. Connecting to Pinecone...")

        pc = Pinecone(api_key=settings.PINECONE_API_KEY)

        create_index_if_missing(pc)

        index = pc.Index(INDEX_NAME)
        print("\nCurrent index stats:")
        print(index.describe_index_stats())

    except Exception:
        print("\nPINECONE SETUP FAILED:")
        traceback.print_exc()
        return

    print("\n2. Verifying the imported ingestion function...")
    print("Function file:", inspect.getsourcefile(PineConeIntegration))
    print(inspect.getsource(PineConeIntegration))

    if not DATA_DIR.is_dir():
        print(f"\nData directory not found: {DATA_DIR.resolve()}")
        return

    files = sorted(DATA_DIR.glob("*.md"))

    if not files:
        print(f"\nNo Markdown files found in {DATA_DIR.resolve()}")
        return

    print(f"\nFound {len(files)} Markdown files.")

    for i, file_path in enumerate(files, start=1):
        print(f"\n[{i}/{len(files)}] File: {file_path}")

        choice = input("Ingest this file? (y/n/q): ").strip().lower()

        if choice == "q":
            print("Stopping ingestion.")
            break

        if choice != "y":
            print("Skipped.")
            continue

        try:
            PineConeIntegration(
                INDEX=INDEX_NAME,
                NAMESPACE=NAMESPACE,
                path=str(file_path),
            )

            print(f"INGESTION SUCCESS: {file_path.name}")

        except Exception:
            print(f"INGESTION FAILED: {file_path.name}")
            traceback.print_exc()

            # Stop at the first failure for easier debugging.
            return

    print("\n3. Checking stored vectors...")

    try:
        stats = pc.Index(INDEX_NAME).describe_index_stats()
        print(stats)

    except Exception:
        traceback.print_exc()
        return

    query = input("\nEnter a retrieval query: ").strip()

    if not query:
        print("No query provided. Exiting.")
        return

    print("\n4. Testing retrieval...")

    try:
        result = get_data_from_pinecone(
            INDEX=INDEX_NAME,
            query=query,
            NAMESPACE=NAMESPACE,
        )

        if result and result.strip():
            print("\nRETRIEVAL SUCCESS")
            print("-" * 60)
            print(result)
        else:
            print(
                "\nNo content retrieved. Verify that ingestion and "
                "retrieval use the same index and namespace."
            )

    except Exception:
        print("\nRETRIEVAL FAILED:")
        traceback.print_exc()


if __name__ == "__main__":
    main()


1. Connecting to Pinecone...
Available indexes: ['hr-policy-rag', 'agentic-rag-kb']
Requested index: 'knowledge-base-customer-assistant'

Index 'knowledge-base-customer-assistant' does not exist.
Creating Pinecone index...

Waiting for index readiness...
Index 'knowledge-base-customer-assistant' is ready.

Current index stats:
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

2. Verifying the imported ingestion function...
Function file: d:\AIProjects\AIBasedSupportSystemWithDashboard\app\ingestion\loading_data.py
def PineConeIntegration(INDEX: str, NAMESPACE: str, path: str):
    existing_list = pinecone.list_indexes().names()

    if INDEX not in existing_list:
        create_index(INDEX, NAMESPACE)

    print(f"Index requested: {INDEX!r}")
    print(f"Source file: {path}")

    docs = load_data_from_path(path)
    chunks = chunking_docs(docs)

    if not chunks:
        raise ValueError(f"No chunks 

In [2]:

import asyncio

from app.guardrails.actions import check_topic_allowed


async def main():
    queries = [
        "what is the refund policy",
        "How can I track my order?",
        "My payment failed",
        "What is the capital of France?",
    ]

    for query in queries:
        context = {"last_user_message": query}
        result = await check_topic_allowed(context)

        print(f"\nQuery: {query}")
        print(f"Allowed: {result}")


await main()


[Guardrails Debug] action=check_topic_allowed allowed=True message_found=True

Query: what is the refund policy
Allowed: True
[Guardrails Debug] action=check_topic_allowed allowed=True message_found=True

Query: How can I track my order?
Allowed: True
[Guardrails Debug] action=check_topic_allowed allowed=True message_found=True

Query: My payment failed
Allowed: True
[Guardrails Debug] action=check_topic_allowed allowed=False message_found=True

Query: What is the capital of France?
Allowed: False


In [3]:
from app.evaluation.evaluate_retrieval import build_ragas_dataset
build_ragas_dataset({
    "id": "account_001",
    "category": "account_security",
    "user_input": "How can I reset my ShopEase password?",
    "reference": "Choose Forgot Password on the sign-in page and follow the instructions sent to your registered contact method. Use only the official ShopEase website or app.",
    "reference_contexts": [
      "Choose Forgot Password on the sign-in page and follow the instructions sent to the registered contact method. Use the official ShopEase website or app only."
    ]
  })

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4294.49it/s]


TypeError: string indices must be integers, not 'str'

In [2]:

import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from ragas.metrics.collections import (
    ContextPrecisionWithReference,
    ContextRecall,
)

load_dotenv()

# 1. Initialize Groq LLM
groq_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

# 2. Wrap LangChain LLM for Ragas
evaluator_llm = LangchainLLMWrapper(groq_llm)

# 3. Initialize retrieval metrics
context_precision = ContextPrecisionWithReference(
    llm=evaluator_llm
)

context_recall = ContextRecall(
    llm=evaluator_llm
)

print("Groq evaluator initialized successfully!")
print("Context Precision:", context_precision.name)
print("Context Recall:", context_recall.name)


C:\Users\vedan\AppData\Local\Temp\ipykernel_5976\1249617355.py:20: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(groq_llm)


ValueError: Collections metrics only support modern InstructorLLM. Found: LangchainLLMWrapper. Use: llm_factory('gpt-4o-mini', client=openai_client)

In [3]:
import inspect
from ragas.llms import llm_factory

print(inspect.signature(llm_factory))

(model: 'str', provider: 'str' = 'openai', client: 't.Optional[t.Any]' = None, adapter: 'str' = 'auto', cache: 't.Optional[CacheInterface]' = None, **kwargs: 't.Any') -> 'InstructorBaseRagasLLM'


In [4]:
import instructor

print("Instructor version:", instructor.__version__)
print("Groq integration:", hasattr(instructor, "from_groq"))


Instructor version: 1.17.0
Groq integration: True


In [5]:
import os
import instructor
from groq import Groq

from ragas.llms import llm_factory
from ragas.metrics.collections import (
    ContextPrecisionWithReference,
    ContextRecall,
)

# 1. Create Groq client
groq_client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# 2. Convert Groq client into Instructor client
instructor_client = instructor.from_groq(groq_client)

# 3. Create Ragas-compatible evaluator
evaluator_llm = llm_factory(
    model="openai/gpt-oss-20b",
    provider="groq",
    client=instructor_client,
)

# 4. Initialize metrics
context_precision = ContextPrecisionWithReference(
    llm=evaluator_llm
)

context_recall = ContextRecall(
    llm=evaluator_llm
)

print("Groq + Ragas evaluator initialized successfully!")

Groq + Ragas evaluator initialized successfully!


In [6]:
import inspect
from ragas import EvaluationDataset, SingleTurnSample, evaluate

print("Evaluate:", inspect.signature(evaluate))
print("Dataset:", inspect.signature(EvaluationDataset))
print("Sample:", inspect.signature(SingleTurnSample))
print("Precision score:", inspect.signature(context_precision.ascore))
print("Recall score:", inspect.signature(context_recall.ascore))

Evaluate: (dataset: 't.Union[Dataset, EvaluationDataset]', metrics: 't.Optional[t.Sequence[Metric]]' = None, llm: 't.Optional[BaseRagasLLM | LangchainLLM]' = None, embeddings: 't.Optional[BaseRagasEmbeddings | BaseRagasEmbedding | LangchainEmbeddings]' = None, experiment_name: 't.Optional[str]' = None, callbacks: 'Callbacks' = None, run_config: 't.Optional[RunConfig]' = None, token_usage_parser: 't.Optional[TokenUsageParser]' = None, raise_exceptions: 'bool' = False, column_map: 't.Optional[t.Dict[str, str]]' = None, show_progress: 'bool' = True, batch_size: 't.Optional[int]' = None, _run_id: 't.Optional[UUID]' = None, _pbar: 't.Optional[tqdm]' = None, return_executor: 'bool' = False, allow_nest_asyncio: 'bool' = True) -> 't.Union[EvaluationResult, Executor]'
Dataset: (samples: 't.List[Sample]', backend: 't.Optional[str]' = None, name: 't.Optional[str]' = None) -> None
Sample: (*, user_input: Optional[str] = None, retrieved_contexts: Optional[List[str]] = None, reference_contexts: Opti